# Imports

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:


from src.config import CITIES, START_DATE, END_DATE, DAILY_VARIABLES
from src.ingestion import fetch_all_cities, fetch_forecast_all_cities

# Create output folders

In [3]:
raw_historical_dir = Path("../data/raw/historical")
raw_forecast_dir = Path("../data/raw/forecast")

raw_historical_dir.mkdir(parents=True, exist_ok=True)
raw_forecast_dir.mkdir(parents=True, exist_ok=True)

print("Historical folder:", raw_historical_dir.resolve())
print("Forecast folder:", raw_forecast_dir.resolve())

Historical folder: C:\Users\viktus\Desktop\for_labs\module5\m5-project-weather-pipeline\data\raw\historical
Forecast folder: C:\Users\viktus\Desktop\for_labs\module5\m5-project-weather-pipeline\data\raw\forecast


# Clean old raw files 

In [4]:
# Clean old raw files before saving new ingestion output

for file in raw_historical_dir.glob("*.parquet"):
    file.unlink()

for file in raw_forecast_dir.glob("*.parquet"):
    file.unlink()

print("Old raw files removed.")

Old raw files removed.


# Quick config check

In [5]:
print("Cities:")
for city in CITIES:
    print(city)

print("\nDate range:")
print(START_DATE, "→", END_DATE)

print("\nDaily variables:")
print(DAILY_VARIABLES)

Cities:
{'name': 'Baku', 'latitude': 40.41, 'longitude': 49.87}
{'name': 'Lankaran', 'latitude': 38.75, 'longitude': 48.85}
{'name': 'Guba', 'latitude': 41.36, 'longitude': 48.51}
{'name': 'Gabala', 'latitude': 40.58, 'longitude': 47.5}
{'name': 'Shaki', 'latitude': 41.11, 'longitude': 47.1}

Date range:
2020-04-27 → 2026-04-27

Daily variables:
['temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration']


# Fetch historical data for all cities

In [6]:
historical_data = fetch_all_cities(
    cities_config=CITIES,
    start_date=START_DATE,
    end_date=END_DATE,
    variables=DAILY_VARIABLES,
)

Fetching historical data for Baku...
Baku: 2192 historical rows
Fetching historical data for Lankaran...
Lankaran: 2192 historical rows
Fetching historical data for Guba...
Guba: 2192 historical rows
Fetching historical data for Gabala...
Gabala: 2192 historical rows
Fetching historical data for Shaki...
Shaki: 2192 historical rows


# Inspect historical data

In [7]:
for city_name, df in historical_data.items():
    print(f"\nCity: {city_name}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Date range: {df['time'].min()} → {df['time'].max()}")
    print("Missing values:")
    print(df.isna().sum())


City: Baku
Rows: 2192
Columns: ['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Date range: 2020-04-27 00:00:00 → 2026-04-27 00:00:00
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
apparent_temperature_max     0
sunshine_duration            0
city                         0
dtype: int64

City: Lankaran
Rows: 2192
Columns: ['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Date range: 2020-04-27 00:00:00 → 2026-04-27 00:00:00
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidit

# Save historical raw files

In [8]:
for city_name, df in historical_data.items():
    file_path = raw_historical_dir / f"{city_name.lower()}_2020_2025.parquet"
    df.to_parquet(file_path, index=False)
    print(f"Saved historical file: {file_path}")

Saved historical file: ..\data\raw\historical\baku_2020_2025.parquet
Saved historical file: ..\data\raw\historical\lankaran_2020_2025.parquet
Saved historical file: ..\data\raw\historical\guba_2020_2025.parquet
Saved historical file: ..\data\raw\historical\gabala_2020_2025.parquet
Saved historical file: ..\data\raw\historical\shaki_2020_2025.parquet


# Verify saved historical files

In [9]:
for file in sorted(raw_historical_dir.glob("*.parquet")):
    print(f"\nFile: {file.name}")
    df_check = pd.read_parquet(file)
    print(df_check.head(3))
    print(df_check.shape)


File: baku_2020_2025.parquet
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2020-04-27                19.3                0.0                15.9   
1 2020-04-28                14.6                0.0                42.0   
2 2020-04-29                17.6                0.0                22.4   

   relative_humidity_2m_mean  cloud_cover_mean  apparent_temperature_max  \
0                         78                26                      19.8   
1                         83                71                      11.9   
2                         61                50                      16.3   

   sunshine_duration  city  
0           45068.90  Baku  
1           38436.84  Baku  
2           39311.92  Baku  
(2192, 9)

File: gabala_2020_2025.parquet
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2020-04-27                23.4                0.0                17.9   
1 2020-04-28                22.1                0.0     

# Fetch forecast data for all cities

In [10]:
forecast_data = fetch_forecast_all_cities(
    cities_config=CITIES,
    variables=DAILY_VARIABLES,
)

Fetching forecast data for Baku...
Baku: 7 forecast rows
Fetching forecast data for Lankaran...
Lankaran: 7 forecast rows
Fetching forecast data for Guba...
Guba: 7 forecast rows
Fetching forecast data for Gabala...
Gabala: 7 forecast rows
Fetching forecast data for Shaki...
Shaki: 7 forecast rows


# Inspect forecast data

In [11]:
for city_name, df in forecast_data.items():
    print(f"\nCity: {city_name}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Date range: {df['time'].min()} → {df['time'].max()}")
    print("Missing values:")
    print(df.isna().sum())


City: Baku
Rows: 7
Columns: ['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Date range: 2026-04-28 00:00:00 → 2026-05-04 00:00:00
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
apparent_temperature_max     0
sunshine_duration            0
city                         0
dtype: int64

City: Lankaran
Rows: 7
Columns: ['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']
Date range: 2026-04-28 00:00:00 → 2026-05-04 00:00:00
Missing values:
time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_m

# Save forecast raw files

In [12]:
for city_name, df in forecast_data.items():
    file_path = raw_forecast_dir / f"{city_name.lower()}_forecast.parquet"
    df.to_parquet(file_path, index=False)
    print(f"Saved forecast file: {file_path}")

Saved forecast file: ..\data\raw\forecast\baku_forecast.parquet
Saved forecast file: ..\data\raw\forecast\lankaran_forecast.parquet
Saved forecast file: ..\data\raw\forecast\guba_forecast.parquet
Saved forecast file: ..\data\raw\forecast\gabala_forecast.parquet
Saved forecast file: ..\data\raw\forecast\shaki_forecast.parquet


# Verify saved forecast files

In [13]:
for file in sorted(raw_forecast_dir.glob("*.parquet")):
    print(f"\nFile: {file.name}")
    df_check = pd.read_parquet(file)
    print(df_check.head(3))
    print(df_check.shape)


File: baku_forecast.parquet
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2026-04-28                14.1                0.0                45.5   
1 2026-04-29                11.4                2.0                26.2   
2 2026-04-30                17.8                0.0                19.7   

   relative_humidity_2m_mean  cloud_cover_mean  apparent_temperature_max  \
0                         51                91                       7.1   
1                         68               100                       6.6   
2                         62                45                      19.1   

   sunshine_duration  city  
0           32312.05  Baku  
1               0.00  Baku  
2           45458.65  Baku  
(7, 9)

File: gabala_forecast.parquet
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2026-04-28                17.4                0.0                19.5   
1 2026-04-29                13.3                0.3          

# Combine historical data into one dataframe

In [14]:
historical_all = pd.concat(historical_data.values(), ignore_index=True)
historical_all = historical_all.sort_values(["city", "time"]).reset_index(drop=True)

print(historical_all.shape)
print(historical_all.head())

(10960, 9)
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2020-04-27                19.3                0.0                15.9   
1 2020-04-28                14.6                0.0                42.0   
2 2020-04-29                17.6                0.0                22.4   
3 2020-04-30                18.0                0.0                23.0   
4 2020-05-01                19.5                0.0                20.2   

   relative_humidity_2m_mean  cloud_cover_mean  apparent_temperature_max  \
0                         78                26                      19.8   
1                         83                71                      11.9   
2                         61                50                      16.3   
3                         77                50                      16.0   
4                         78                24                      18.5   

   sunshine_duration  city  
0           45068.90  Baku  
1           38436.84  B

# Combine forecast data into one dataframe

In [15]:
forecast_all = pd.concat(forecast_data.values(), ignore_index=True)
forecast_all = forecast_all.sort_values(["city", "time"]).reset_index(drop=True)

print(forecast_all.shape)
print(forecast_all.head())

(35, 9)
        time  temperature_2m_max  precipitation_sum  wind_speed_10m_max  \
0 2026-04-28                14.1                0.0                45.5   
1 2026-04-29                11.4                2.0                26.2   
2 2026-04-30                17.8                0.0                19.7   
3 2026-05-01                17.6                0.0                35.1   
4 2026-05-02                19.4                0.0                27.0   

   relative_humidity_2m_mean  cloud_cover_mean  apparent_temperature_max  \
0                         51                91                       7.1   
1                         68               100                       6.6   
2                         62                45                      19.1   
3                         81                33                      14.0   
4                         80                42                      17.2   

   sunshine_duration  city  
0           32312.05  Baku  
1               0.00  Baku

# Final raw-data sanity checks

In [16]:
print("Historical duplicate rows:", historical_all.duplicated().sum())
print("Forecast duplicate rows:", forecast_all.duplicated().sum())

print("\nHistorical date range:", historical_all["time"].min(), "→", historical_all["time"].max())
print("Forecast date range:", forecast_all["time"].min(), "→", forecast_all["time"].max())

print("\nHistorical columns:")
print(historical_all.columns.tolist())

print("\nForecast columns:")
print(forecast_all.columns.tolist())

Historical duplicate rows: 0
Forecast duplicate rows: 0

Historical date range: 2020-04-27 00:00:00 → 2026-04-27 00:00:00
Forecast date range: 2026-04-28 00:00:00 → 2026-05-04 00:00:00

Historical columns:
['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']

Forecast columns:
['time', 'temperature_2m_max', 'precipitation_sum', 'wind_speed_10m_max', 'relative_humidity_2m_mean', 'cloud_cover_mean', 'apparent_temperature_max', 'sunshine_duration', 'city']


# DuckDB Loading

In [17]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [18]:
from src.db import create_schemas, load_raw_data, run_query

create_schemas()
load_raw_data()

run_query("SELECT * FROM raw.historical LIMIT 5")

,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city
0,2020-04-27,19.3,0.0,15.9,78,26,19.8,45068.90,Baku
1,2020-04-28,14.6,0.0,42.0,83,71,11.9,38436.84,Baku
2,2020-04-29,17.6,0.0,22.4,61,50,16.3,39311.92,Baku
3,2020-04-30,18.0,0.0,23.0,77,50,16.0,44363.33,Baku
4,2020-05-01,19.5,0.0,20.2,78,24,18.5,46341.87,Baku


In [19]:
run_query("""
SELECT table_schema, table_name
FROM information_schema.tables
""")

,table_schema,table_name
0,analytics,future_28d_forecast
1,analytics,model_features
2,raw,forecast
3,raw,historical
